# Extract dialogue from *Harry Potter and the Philosopher’s Stone*

This notebook converts `book_data/hp1_book.txt` into a CSV with the same columns as `movie_data/hp1.csv`: `movie`, `chapter`, `character`, `dialog`.

The source is a prose novel, not a transcript. The extraction identifies text inside quotation marks and uses nearby “said/asked/replied…” attribution patterns to guess the speaker. Book prose often leaves speakers implicit or uses action beats, so this is a best-effort, reviewable extraction: rows assigned `Unknown` and the sampled output should be checked before using the table for network analysis. The code keeps each quoted speech passage as one dialogue row.

In [12]:
from pathlib import Path
import re
import pandas as pd

BOOK_PATH = Path("../book_data/hp1_book.txt")
MOVIE_CSV = Path("../movie_data/hp1.csv")
OUTPUT_CSV = Path("../book_data/hp1_dialogue.csv")

text = BOOK_PATH.read_text(encoding="utf-8-sig")
movie_title = pd.read_csv(MOVIE_CSV)["movie"].iloc[0]
print(f"Read {len(text):,} characters; movie column will use {movie_title!r}.")
print(text[:500])


Read 439,741 characters; movie column will use "Harry Potter and the Philosopher's Stone".
Harry Potter and the Sorcerer's Stone


CHAPTER ONE

THE BOY WHO LIVED

Mr. and Mrs. Dursley, of number four, Privet Drive, were proud to say
that they were perfectly normal, thank you very much. They were the last
people you'd expect to be involved in anything strange or mysterious,
because they just didn't hold with such nonsense.

Mr. Dursley was the director of a firm called Grunnings, which made
drills. He was a big, beefy man with hardly any neck, although he did
have a very large mustache


## Split the text into chapters and dialogue passages

Chapter headings are normalized to chapter names. The first occurrence of a chapter title is used, and quoted passages retain their source order.

In [13]:
def chapter_sections(book_text):
    # Match headings such as CHAPTER ONE, including Roman/ordinal/numeric forms.
    matches = list(re.finditer(r"(?im)^\s*CHAPTER\s+([^\n]+)\s*$", book_text))
    if not matches:
        raise ValueError("No CHAPTER headings found in the input text.")
    sections = []
    for i, match in enumerate(matches):
        end = matches[i + 1].start() if i + 1 < len(matches) else len(book_text)
        raw_title = match.group(1).strip()
        body_start = match.end()
        body = book_text[body_start:end]
        # The title may be on the next non-empty line (as in this source).
        lines = body.splitlines()
        title = raw_title
        first_content = 0
        for j, line in enumerate(lines):
            if line.strip():
                title = line.strip()
                first_content = j + 1
                break
        body = "\n".join(lines[first_content:])
        sections.append((title, body))
    return sections

sections = chapter_sections(text)
print(f"Found {len(sections)} chapters; first chapters: {[name for name, _ in sections[:5]]}")


Found 17 chapters; first chapters: ['THE BOY WHO LIVED', 'THE VANISHING GLASS', 'THE LETTERS FROM NO ONE', 'THE KEEPER OF THE KEYS', 'DIAGON ALLEY']


## Extract dialogue and attribute it from speech tags

The extractor uses speech tags such as `“...,” said Harry`, `Harry said, “...”`, and `... Harry said` (including unquoted text immediately before the tag). It strips the attribution phrase from the dialogue text. Multiple speech turns in a paragraph are handled separately where tags identify them. Quoted passages with no usable nearby tag remain as `Unknown` so they can be reviewed; the parser does not invent a speaker.

In [14]:
CHARACTER_NAMES = [
    "Albus Dumbledore", "Minerva McGonagall", "Rubeus Hagrid", "Harry Potter",
    "Hermione Granger", "Ron Weasley", "Draco Malfoy", "Neville Longbottom",
    "Severus Snape", "Quirinus Quirrell", "Voldemort", "Lord Voldemort",
    "Petunia Dursley", "Vernon Dursley", "Dudley Dursley", "Oliver Wood",
    "Percy Weasley", "Fred Weasley", "George Weasley", "Molly Weasley",
    "Lee Jordan", "Argus Filch", "Nearly Headless Nick", "Sorting Hat",
    "Madam Hooch", "Professor McGonagall", "Professor Flitwick",
    "Professor Sprout", "Madam Pomfrey", "Ollivander", "Griphook",
    "Dedalus Diggle", "Aunt Marge", "Firenze", "Bane", "Fang",
]
SPEECH_VERBS = r"(?:said|asked|replied|whispered|shouted|yelled|cried|called|muttered|answered|exclaimed|snapped|barked|groaned|moaned|laughed|gasped|continued|added|interrupted|demanded|suggested|stammered|sobbed|hissed|screamed|began|went on|told|remarked|insisted|inquired)"
name_pattern = "|".join(re.escape(name) for name in sorted(CHARACTER_NAMES, key=len, reverse=True))

# The tag can be either after the quote (said Harry) or before it (Harry said).
TAG_AFTER = re.compile(rf"(?is)(?P<pre>.*?)(?:[”\"]\s*)?(?:,?\s+)\b{SPEECH_VERBS}\b(?:\s+to)?\s+(?P<name>{name_pattern})\b(?P<tail>[^.!?\n]*)(?=[.!?]|$)")
TAG_BEFORE = re.compile(rf"(?is)\b(?P<name>{name_pattern})\b(?:[^.!?\n]{{0,35}}?)\b{SPEECH_VERBS}\b\s*[:,]?\s*(?P<post>[“\"][^”\"]*[”\"])?")
QUOTE_RE = re.compile(r"[“\"](.*?)[”\"]", re.DOTALL)

def clean_speech(text):
    text = re.sub(r"^[\s,;:—–-]+|[\s,;:—–-]+$", "", text)
    text = re.sub(r"^[“\"]|[”\"]$", "", text).strip()
    return re.sub(r"\s+", " ", text)

def extract_paragraph_dialogue(paragraph):
    """Extract one or more tagged turns in a paragraph, omitting speech tags."""
    rows = []
    consumed = []
    # Tags following dialogue. Take text since the previous tag/paragraph boundary,
    # preferring the final quoted span; when no quote exists, use the preceding clause.
    for match in TAG_AFTER.finditer(paragraph):
        prefix = paragraph[match.start("pre"):match.start() + len(match.group("pre"))]
        # Use the text since the last completed attribution, avoiding earlier turns.
        start = max(prefix.rfind(" said "), prefix.rfind(" asked "), prefix.rfind(" replied "), prefix.rfind("?”"), prefix.rfind("!”"), prefix.rfind(".”"))
        candidate = prefix[start + 1:] if start >= 0 else prefix
        quoted = list(QUOTE_RE.finditer(candidate))
        speech = quoted[-1].group(1) if quoted else candidate
        speech = clean_speech(speech)
        if speech:
            rows.append((match.group("name"), speech))
            consumed.append((max(0, match.start("pre") - len(candidate)), match.end()))
    # Tags before dialogue. These typically introduce the quote following the tag.
    for match in TAG_BEFORE.finditer(paragraph):
        post = match.group("post")
        if post:
            speech = clean_speech(post)
            if speech:
                rows.append((match.group("name"), speech))
    if rows:
        return rows
    # Keep stand-alone quotations for manual review when no speaker tag was found.
    return [("Unknown", clean_speech(m.group(1))) for m in QUOTE_RE.finditer(paragraph) if clean_speech(m.group(1))]

def extract_quotes(chapter_text):
    rows = []
    for paragraph in re.split(r"\n\s*\n", chapter_text):
        rows.extend(extract_paragraph_dialogue(paragraph.strip()))
    return rows


In [15]:
records = []
for chapter, body in sections:
    for character, utterance in extract_quotes(body):
        records.append({"movie": movie_title, "chapter": chapter, "character": character, "dialog": utterance})

book_dialogue = pd.DataFrame(records, columns=["movie", "chapter", "character", "dialog"])
print(f"Extracted {len(book_dialogue):,} quoted passages")
print(book_dialogue["character"].value_counts().head(15))
book_dialogue.head(20)


Extracted 2,300 quoted passages
character
Unknown                 2273
Professor McGonagall      11
Griphook                   4
Firenze                    3
Professor Flitwick         2
Dedalus Diggle             1
Ollivander                 1
Nearly Headless Nick       1
Madam Hooch                1
Hermione Granger           1
Fred Weasley               1
Voldemort                  1
Name: count, dtype: int64


,movie,chapter,character,dialog
0,Harry Potter and the Philosopher's Stone,THE BOY WHO LIVED,Unknown,Little tyke
1,Harry Potter and the Philosopher's Stone,THE BOY WHO LIVED,Unknown,"The Potters, that's right, that's what I heard..."
2,Harry Potter and the Philosopher's Stone,THE BOY WHO LIVED,Unknown,Sorry
3,Harry Potter and the Philosopher's Stone,THE BOY WHO LIVED,Unknown,"Don't be sorry, my dear sir, for nothing could..."
4,Harry Potter and the Philosopher's Stone,THE BOY WHO LIVED,Unknown,Shoo!
5,Harry Potter and the Philosopher's Stone,THE BOY WHO LIVED,Unknown,Won't!
6,Harry Potter and the Philosopher's Stone,THE BOY WHO LIVED,Unknown,"And finally, bird-watchers everywhere have rep..."
7,Harry Potter and the Philosopher's Stone,THE BOY WHO LIVED,Unknown,"Most mysterious. And now, over to Jim McGuffin..."
8,Harry Potter and the Philosopher's Stone,THE BOY WHO LIVED,Unknown,"Well, Ted"
9,Harry Potter and the Philosopher's Stone,THE BOY WHO LIVED,Unknown,"I don't know about that, but it's not only the..."


## Review and export

Inspect rows marked `Unknown` and a few speaker transitions before using the file for analysis. The output schema and column order match the movie CSV exactly. Chapters are the book's chapter titles; the `movie` field is copied from the first movie CSV so the label remains consistent with the existing dataset.

In [16]:
print("Unknown speaker rows:", int((book_dialogue["character"] == "Unknown").sum()))
book_dialogue.to_csv(OUTPUT_CSV, index=False, encoding="utf-8")
print(f"Wrote {len(book_dialogue):,} rows to {OUTPUT_CSV.resolve()}")
print("Columns:", book_dialogue.columns.tolist())


Unknown speaker rows: 2273
Wrote 2,300 rows to /Users/monicahardt/Desktop/02805-SocialGraphs-RAM/book_data/hp1_dialogue.csv
Columns: ['movie', 'chapter', 'character', 'dialog']
